### Aula 10 — Performance: plano, distribuição e reutilização

Esta aula investiga os custos que aparecem nos planos: filtros locais, shuffle, agregações, skew, estratégias de join e cache. A base é sintética, criada com `spark.range`; nenhuma entrada externa é necessária. É preciso um ambiente PySpark capaz de processar 2 milhões de linhas e, para investigação interativa adicional, acesso à Spark UI da aplicação.

**Como ler:** os números abaixo são os outputs preservados. Um operador no plano explica a estratégia escolhida; a ausência de durações, bytes transferidos ou métricas de tasks nos outputs desta aula impede declarar uma estratégia mais rápida apenas por sua aparência. A ordem dos experimentos foi mantida.

**Referência de estudo:** o [capítulo 10 — Performance, diagnóstico e otimização](../docs/10_performance.md) aprofunda os experimentos desta aula e distingue seus resultados dos registros complementares do desafio 10.

#### 1. Iniciar a aplicação

`SparkSession` cria ou reutiliza a sessão; `F` fornece expressões executadas pelo Spark. O aviso sobre a porta 4040 indica que a UI tentou usar 4041. Ele não registra gargalo do processamento.


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_10_performance")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/17 23:46:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/17 23:46:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


#### 2. Criar uma base distribuída controlada

`spark.range(0, 2_000_000, 1, numPartitions=8)` define a coluna `id` com valores de 0 a 1.999.999 em oito partições de entrada. A transformação descreve uma fonte geradora; o processamento ocorre conforme as actions exigem dados.

Spark Partitions são divisões do trabalho, tasks processam essas divisões e executors fornecem recursos para executá-las. O número de partições não equivale ao número de executors nem garante oito tasks simultâneas. Essa base regular permite observar como filtros e redistribuições mudam o trabalho sem confundir o experimento com leitura de arquivos.


In [3]:
df_base = spark.range(
    start=0,
    end=2_000_000,
    step=1,
    numPartitions=8
)

#### 3. Filtro narrow: menos linhas, mesmo número de partições

O filtro `id % 10 == 0` retém um em cada dez IDs. **Observado:** `count()` retorna 200.000 e `getNumPartitions()` continua em 8. Reduzir linhas não reduz automaticamente o número de divisões de execução.

No `explain(True)`, o plano físico mostra `Filter` diretamente sobre `Range`, sem `Exchange`. Cada partição pode avaliar a condição sobre suas próprias linhas: trata-se de dependência narrow neste caso. Os símbolos `*(1)` identificam operadores no mesmo estágio de geração de código, não uma contagem de executors. O plano não contém leitura externa nem shuffle para esse filtro.


In [4]:
df_filtrado = (
    df_base.filter(F.col("id") % 10 == 0)
)

In [5]:
df_filtrado.count()

200000

In [6]:
df_filtrado.rdd.getNumPartitions()

8

In [7]:
df_filtrado.explain(True)

== Parsed Logical Plan ==
'Filter (('id % 10) = 0)
+- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Filter ((id#0L % cast(10 as bigint)) = cast(0 as bigint))
+- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Filter ((id#0L % 10) = 0)
+- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
*(1) Filter ((id#0L % 10) = 0)
+- *(1) Range (0, 2000000, step=1, splits=8)



#### 4. Agregação wide e redução parcial

Apesar do nome `df_filtrado2`, este DataFrame parte de `df_base`, e não dos 200.000 registros filtrados. `withColumn` cria `grupo = id % 100`; há 100 grupos possíveis sobre os 2 milhões de IDs. As dez linhas mostradas têm contagem 20.000 cada. Sem `orderBy`, a ordem apresentada não é garantida.

**Plano físico salvo:** `Range → Project → HashAggregate partial_count → Exchange hashpartitioning(grupo, 200) → HashAggregate count`. A etapa parcial conta por chave localmente, reduzindo os dados enviados; o `Exchange` redistribui os resultados para reunir a mesma chave; a etapa final consolida as contagens. É uma dependência wide pela redistribuição entre partições.

`AdaptiveSparkPlan isFinalPlan=false` indica um plano adaptativo ainda não finalizado nesse registro. Os 200 destinos de shuffle exibidos não provam a quantidade final de tasks após AQE. O `Project` otimizado mantém somente `grupo`, pois `id` deixa de ser necessário para a agregação.


In [8]:
df_filtrado2 = (
    df_base.withColumn("grupo", (F.col("id") % 100))
)

In [9]:
df_agrupado = (
    df_filtrado2.groupBy("grupo")
    .count()
)

In [10]:
df_agrupado.show(10)

+-----+-----+
|grupo|count|
+-----+-----+
|   26|20000|
|   29|20000|
|   65|20000|
|   19|20000|
|   54|20000|
|    0|20000|
|   22|20000|
|    7|20000|
|   77|20000|
|   34|20000|
+-----+-----+
only showing top 10 rows



In [11]:
df_agrupado.explain(True)

== Parsed Logical Plan ==
'Aggregate ['grupo], ['grupo, count(1) AS count#16L]
+- Project [id#0L, (id#0L % cast(100 as bigint)) AS grupo#10L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
grupo: bigint, count: bigint
Aggregate [grupo#10L], [grupo#10L, count(1) AS count#16L]
+- Project [id#0L, (id#0L % cast(100 as bigint)) AS grupo#10L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Aggregate [grupo#10L], [grupo#10L, count(1) AS count#16L]
+- Project [(id#0L % 100) AS grupo#10L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[grupo#10L], functions=[count(1)], output=[grupo#10L, count#16L])
   +- Exchange hashpartitioning(grupo#10L, 200), ENSURE_REQUIREMENTS, [plan_id=107]
      +- HashAggregate(keys=[grupo#10L], functions=[partial_count(1)], output=[grupo#10L, count#26L])
         +- Project [(id#0L % 100) AS grupo#10L]
            +- Range (0,

#### 5. `repartition(4)`: redistribuir com shuffle

O notebook redefine a distribuição da base original em quatro partições. **Observado:** `getNumPartitions()` retorna 4 e o plano final contém `ShuffleQueryStage` com `Exchange RoundRobinPartitioning(4)`.

Sem uma chave, a redistribuição é round-robin. Ela pode equilibrar o trabalho, mas exige movimentação entre partições mesmo quando o objetivo é reduzir de oito para quatro. Neste registro o plano adaptativo está finalizado (`isFinalPlan=true`); os blocos Initial e Final permitem comparar a estratégia antes e depois da execução correspondente.


In [12]:
df_reparticionado = (df_base.repartition(4))

In [13]:
df_reparticionado.rdd.getNumPartitions()

4

In [14]:
df_reparticionado.explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Repartition 4, true
+- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Repartition 4, true
+- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=120]
      +- *(1) Range (0, 2000000, step=1, splits=8)
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=115]
   +- Range (0, 2000000, step=1, splits=8)



#### 6. `coalesce(4)`: reunir partições existentes

Também reduz a base a quatro partições, mas o plano mostra `Coalesce 4` sobre `Range`, sem `Exchange` adicional. A operação pode combinar partições existentes sem redistribuir cada linha por shuffle.

Essa economia potencial de movimentação tem um compromisso: `coalesce` não reequilibra arbitrariamente os dados e pode reduzir demais o paralelismo de etapas anteriores. A escolha depende do estado das partições e do próximo trabalho; quatro partições obtidas pelos dois métodos não significam o mesmo custo.


In [15]:
df_coalesce = (df_base.coalesce(4))

In [16]:
df_coalesce.rdd.getNumPartitions()

4

In [17]:
df_coalesce.explain(True)

== Parsed Logical Plan ==
Repartition 4, false
+- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Repartition 4, false
+- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Repartition 4, false
+- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
Coalesce 4
+- *(1) Range (0, 2000000, step=1, splits=8)



##### 6.1. Comparar o volume por partição

Nos dois blocos, `spark_partition_id()` é calculado antes do agrupamento diagnóstico. Assim, a contagem descreve a distribuição de `df_reparticionado` e `df_coalesce` naquele ponto; a própria agregação pode criar novas partições depois.

**Observado:** ambos apresentam quatro partições de 500.000 linhas. A origem regular permitiu equilíbrio mesmo com `coalesce`. Isso não garante comportamento igual para uma origem desigual, nem comprova empate de duração. Contagens equilibradas também não significam igual custo por linha quando as operações são diferentes.


In [18]:
(
    df_reparticionado.withColumn("particoes", F.spark_partition_id())
    .groupBy("particoes")
    .agg(F.count("particoes").alias("quantidade_registros"))
    .orderBy("particoes")
).show()

+---------+--------------------+
|particoes|quantidade_registros|
+---------+--------------------+
|        0|              500000|
|        1|              500000|
|        2|              500000|
|        3|              500000|
+---------+--------------------+



In [19]:
(
    df_coalesce.withColumn("particoes", F.spark_partition_id())
    .groupBy("particoes")
    .agg(F.count("particoes").alias("quantidade_registros"))
    .orderBy("particoes")
).show()

+---------+--------------------+
|particoes|quantidade_registros|
+---------+--------------------+
|        0|              500000|
|        1|              500000|
|        2|              500000|
|        3|              500000|
+---------+--------------------+



#### 7. Construir uma chave com skew

`when(id < 1800000, 0)` atribui 90% dos registros ao grupo 0; os demais recebem grupos de 1 a 9. A contagem salva confirma 2.000.000 linhas. `df_resumo` define uma agregação, mas não a executa nem exibe seu resultado neste bloco.

O skew é a concentração desigual de dados ou trabalho. Uma chave muito frequente pode concentrar uma etapa que precisa reunir chaves iguais, embora a base ainda esteja inicialmente distribuída por faixas regulares de IDs. O desequilíbrio da chave e o desequilíbrio das partições são observações relacionadas, mas não equivalentes.


In [20]:
df_skew = (
    df_base.withColumn("grupo", F.when(F.col("id") < 1800000, 0).otherwise((F.col("id") % 9) + 1))
    
)

In [21]:
df_skew.count()

2000000

In [22]:
df_resumo = (
    df_skew.groupBy("grupo")
    .count()
)

##### 7.1. Reparticionar pela chave concentrada

`repartition(8, "grupo")` usa hash da chave; registros com o mesmo grupo precisam ir ao mesmo destino dessa redistribuição. **Observado:** existem oito partições e o plano final confirma `Exchange hashpartitioning(grupo, 8)`.

Ter oito partições não divide o grupo 0 em oito partes. Chaves distintas também podem cair na mesma partição por hash, e algumas partições podem ficar vazias. Portanto, aumentar o número de destinos sem tratar a concentração não garante equilíbrio. Este é o custo que a próxima contagem investiga.


In [23]:
df_skew_reparticionado = df_skew.repartition(8, "grupo")

In [24]:
df_skew_reparticionado.rdd.getNumPartitions()

8

In [25]:
df_skew_reparticionado.explain(True)

== Parsed Logical Plan ==
'RepartitionByExpression ['grupo], 8
+- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint, grupo: bigint
RepartitionByExpression [grupo#74L], 8
+- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
RepartitionByExpression [grupo#74L], 8
+- Project [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange hashpartitioning(grupo#74L, 8), REPARTITION_BY_NUM, [plan_id=360]
      +- *(1) Project [id#0L, CASE

##### 7.2. Identificar a concentração real

A contagem registra somente as partições que receberam linhas: IDs 0, 1, 3, 4 e 5. A partição 5 contém **1.844.445** registros; as demais exibidas têm 22.223, 22.222, 44.444 e 66.666. Os IDs ausentes não significam redução das oito partições: partições vazias não produzem grupo na contagem.

A maior partição reúne o grupo dominante e outros grupos que compartilham seu destino de hash. Isso pode produzir uma task muito mais longa, mas a tabela mede linhas, não duração ou spill. Como a atribuição termina em `.show()`, `df_desequilibrio` recebe o retorno dessa action (`None`), não um novo DataFrame.


In [26]:
df_desequilibrio = (
    df_skew_reparticionado.withColumn("quantidade_particoes", F.spark_partition_id())
    .groupBy("quantidade_particoes")
    .agg(F.count("quantidade_particoes").alias("registros"))
).show()

+--------------------+---------+
|quantidade_particoes|registros|
+--------------------+---------+
|                   0|    22223|
|                   1|    22222|
|                   3|    44444|
|                   4|    66666|
|                   5|  1844445|
+--------------------+---------+



##### 7.3. Redistribuir sem chave e reconhecer o limite

`repartition(8)` troca o particionamento por chave por round-robin. **Observado:** as oito partições têm 250.000 linhas, e o plano mostra `Exchange RoundRobinPartitioning(8)`. `df_desequilibrio2` também recebe o retorno de `show()`, não a consulta.

Essa distribuição equilibra o volume naquele ponto, ao custo de shuffle. A chave continua com 1,8 milhão de ocorrências no grupo 0; um join ou outra etapa que volte a exigir distribuição por `grupo` pode concentrá-las novamente. Redistribuir sem chave não elimina o skew lógico dos dados.


In [27]:
df_skew_reparticionado2 = df_skew.repartition(8)

In [28]:
df_desequilibrio2 = (
    df_skew_reparticionado2.withColumn("quantidade_particoes", F.spark_partition_id())
    .groupBy("quantidade_particoes")
    .agg(F.count("quantidade_particoes").alias("registros"))
).show()

+--------------------+---------+
|quantidade_particoes|registros|
+--------------------+---------+
|                   0|   250000|
|                   1|   250000|
|                   2|   250000|
|                   3|   250000|
|                   4|   250000|
|                   5|   250000|
|                   6|   250000|
|                   7|   250000|
+--------------------+---------+



In [29]:
df_skew_reparticionado2.explain(True)

== Parsed Logical Plan ==
Repartition 8, true
+- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint, grupo: bigint
Repartition 8, true
+- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Repartition 8, true
+- Project [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
   +- Range (0, 2000000, step=1, splits=Some(8))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=559]
   +- Project [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
      +- Range (0, 2000000, step=1, split

#### 8. Agregar dados com chave concentrada

`df_agregacao_skew` usa `df_skew` original e calcula `count(grupo)`, ordenando o resumo por chave. Como os grupos gerados não são nulos, espera-se a mesma contagem de linhas por grupo. Este bloco salva o plano, sem exibir o resultado dessa variável.

O plano contém duas redistribuições com finalidades distintas: `HashAggregate partial_count → Exchange hashpartitioning → HashAggregate count` reúne contagens por chave; depois `Exchange rangepartitioning → Sort` sustenta a ordenação global solicitada por `orderBy`.

A agregação parcial é particularmente relevante: o shuffle de contagens parciais não precisa transferir 1,8 milhão de linhas brutas do grupo 0. Por isso, skew de chave não autoriza concluir que toda agregação terá o mesmo gargalo de um join por chave. O plano ainda registra `isFinalPlan=false`; não há métricas de tasks dessa consulta aqui.


In [30]:
df_agregacao_skew = (
    df_skew.groupBy("grupo")
    .agg(F.count("grupo").alias("registros_grupo"))
    .orderBy("grupo")
)

In [31]:
df_agregacao_skew.explain(True)

== Parsed Logical Plan ==
'Sort ['grupo ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS registros_grupo#137L]
   +- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
      +- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
grupo: bigint, registros_grupo: bigint
Sort [grupo#74L ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS registros_grupo#137L]
   +- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
      +- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Sort [grupo#74L ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS registros_grupo#137L]
   +- Project [CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
  

#### 9. Criar uma dimensão pequena

A lista contém dez pares únicos de `grupo` e `descricao`, convertidos em `df_dim_grupos`. Essa dimensão cobre os dez grupos gerados na base. A unicidade é importante: neste cenário, o join inner deve associar uma descrição por linha sem multiplicar registros. Uma dimensão com grupos repetidos mudaria a cardinalidade e precisaria de validação própria.


In [32]:
dados = [
    (0, "Grupo 0"),
    (1, "Grupo 1"),
    (2, "Grupo 2"),
    (3, "Grupo 3"),
    (4, "Grupo 4"),
    (5, "Grupo 5"),
    (6, "Grupo 6"),
    (7, "Grupo 7"),
    (8, "Grupo 8"),
    (9, "Grupo 9")
]

In [33]:
df_dim_grupos = spark.createDataFrame(
    dados,
    schema= ["grupo", "descricao"]
)

##### 9.1. BroadcastHashJoin com a dimensão à direita

`F.broadcast(df_dim_grupos)` sinaliza que a dimensão pode ser distribuída aos executors para construir uma estrutura hash local. As partições da base maior procuram correspondências nessa estrutura, evitando a redistribuição da base por `grupo` para este join.

**Observado no plano:** `BroadcastExchange` prepara/distribui a dimensão; `BroadcastHashJoin ... Inner, BuildRight` confirma que o lado direito é usado para construir a relação hash. `Project` escolhe as colunas da saída e os filtros de nulos tratam as chaves do inner join. Não aparece um `Exchange hashpartitioning` da base principal neste join.

Broadcast continua tendo custo de distribuição e memória; a dimensão pequena justifica investigá-lo aqui. A concentração no grupo 0 permanece nos dados, mas não exige reunir todas as suas linhas numa partição para realizar essa correspondência.


In [34]:
df_join = df_skew.join(
    F.broadcast(df_dim_grupos),
    on="grupo",
    how="inner"
)

In [35]:
df_join.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [grupo])
:- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
:  +- Range (0, 2000000, step=1, splits=Some(8))
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [grupo#142L, descricao#143], false

== Analyzed Logical Plan ==
grupo: bigint, id: bigint, descricao: string
Project [grupo#74L, id#0L, descricao#143]
+- Join Inner, (grupo#74L = grupo#142L)
   :- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
   :  +- Range (0, 2000000, step=1, splits=Some(8))
   +- ResolvedHint (strategy=broadcast)
      +- LogicalRDD [grupo#142L, descricao#143], false

== Optimized Logical Plan ==
Project [grupo#74L, id#0L, descricao#143]
+- Join Inner, (grupo#74L = grupo#142L), rightHint=(strategy=broadcast)
   :- Project [id#0L, CASE WHEN (id#

##### 9.2. Validar distribuição e cardinalidade do join

**Observado:** `df_join` tem oito partições, cada uma com 250.000 linhas no diagnóstico, totalizando 2.000.000. O filtro por grupo 0 retorna 1.800.000, preservando a concentração da chave. A dimensão tem uma correspondência por grupo, coerente com a manutenção do total de linhas.

O equilíbrio foi herdado da distribuição regular de `df_skew` por IDs: o broadcast evitou redistribuí-lo pela chave concentrada. Isso não significa que broadcast reequilibraria uma entrada já desigual. Essas verificações sustentam a cardinalidade e a distribuição desta consulta. Elas não medem tempo e não significam que um agrupamento posterior por `grupo` terá partições igualmente equilibradas. `df_join_teste`, terminado em `.show()`, recebe `None`, conforme as outras atribuições diagnósticas da aula.


In [36]:
df_join.rdd.getNumPartitions()

8

In [37]:
df_join_teste = (
    df_join.withColumn("particoes", F.spark_partition_id())
    .groupBy("particoes")
    .agg(F.count("particoes").alias("quantidade_registros"))
).show()

+---------+--------------------+
|particoes|quantidade_registros|
+---------+--------------------+
|        0|              250000|
|        1|              250000|
|        2|              250000|
|        3|              250000|
|        4|              250000|
|        5|              250000|
|        6|              250000|
|        7|              250000|
+---------+--------------------+



In [38]:
df_join.count()

2000000

In [39]:
df_join.filter(F.col("grupo") == 0).count()

1800000

##### 9.3. Ler o plano final e as estatísticas de execução

`explain("formatted")` detalha cada operador e apresenta Initial e Final Plan. **Observado:** `AdaptiveSparkPlan` está com `isFinalPlan=true`, o join permanece `BroadcastHashJoin BuildRight` e o `BroadcastQueryStage` traz `rowCount=10` e `sizeInBytes=1024.1 KiB`.

São estatísticas desse estágio de broadcast, não uma medição de tamanho do CSV, do DataFrame inteiro ou da duração da consulta. A presença do estágio adaptativo não demonstra que o AQE trocou a estratégia: o plano inicial já usava broadcast explícito.


In [40]:
df_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Final Plan ==
   * Project (9)
   +- * BroadcastHashJoin Inner BuildRight (8)
      :- * Project (3)
      :  +- * Filter (2)
      :     +- * Range (1)
      +- BroadcastQueryStage (7), Statistics(sizeInBytes=1024.1 KiB, rowCount=10)
         +- BroadcastExchange (6)
            +- * Filter (5)
               +- * Scan ExistingRDD (4)
+- == Initial Plan ==
   Project (12)
   +- BroadcastHashJoin Inner BuildRight (11)
      :- Project (3)
      :  +- Filter (2)
      :     +- Range (1)
      +- BroadcastExchange (10)
         +- Filter (5)
            +- Scan ExistingRDD (4)


(1) Range [codegen id : 2]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Filter [codegen id : 2]
Input [1]: [id#0L]
Condition : CASE WHEN (id#0L < 1800000) THEN true ELSE isnotnull(((id#0L % 9) + 1)) END

(3) Project [codegen id : 2]
Output [2]: [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
Inpu

#### 10. Comparar com SortMergeJoin

Os dois thresholds de broadcast automático, comum e adaptativo, são alterados para `-1`. O novo join usa a dimensão com `hint("merge")`, sem o broadcast explícito da consulta anterior. As inspeções de `.columns` apenas confirmam as chaves e colunas disponíveis.

**Observado no plano formatted:** há `Exchange hashpartitioning(grupo, 200)` nos dois lados, seguido de `Sort` e `SortMergeJoin Inner`. A redistribuição reúne chaves compatíveis e a ordenação permite percorrer os dois lados para fazer as correspondências. O custo potencial inclui rede, ordenação e concentração da chave dominante.

Esse registro tem `isFinalPlan=false`; esta sequência da aula não salva contagem nem métricas finais desse join. Ela demonstra estratégias físicas diferentes, sem provar qual foi mais rápida ou que o AQE corrigiu o skew. As configurações alteradas permanecem válidas na sessão até serem modificadas ou a aplicação encerrar. O capítulo 10 recomenda guardar e restaurar os valores anteriores em novas comparações; essa restauração não aparece nas células originais desta aula e não deve ser registrada como executada.


In [41]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)

In [42]:
df_skew.columns

['id', 'grupo']

In [43]:
df_dim_grupos.columns


['grupo', 'descricao']

In [44]:
df_sort_merge_join = df_skew.join(
    df_dim_grupos.hint("merge"),
    on="grupo",
    how= "inner"
)

In [45]:
df_sort_merge_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Filter (7)
               +- Scan ExistingRDD (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(8))

(2) Filter
Input [1]: [id#0L]
Condition : CASE WHEN (id#0L < 1800000) THEN true ELSE isnotnull(((id#0L % 9) + 1)) END

(3) Project
Output [2]: [id#0L, CASE WHEN (id#0L < 1800000) THEN 0 ELSE ((id#0L % 9) + 1) END AS grupo#74L]
Input [1]: [id#0L]

(4) Exchange
Input [2]: [id#0L, grupo#74L]
Arguments: hashpartitioning(grupo#74L, 200), ENSURE_REQUIREMENTS, [plan_id=1059]

(5) Sort
Input [2]: [id#0L, grupo#74L]
Arguments: [grupo#74L ASC NULLS FIRST], false, 0

(6) Scan ExistingRDD
Output [2]: [grupo#142L, descricao#143]
Arguments: [grupo#142L, descricao#143], MapPartitionsRDD[98] at

#### 11. AQE: interpretar o que foi confirmado

A consulta de `spark.sql.adaptive.enabled` retorna `'true'`. Adaptive Query Execution pode ajustar decisões com estatísticas obtidas durante a execução, como organização de shuffle e estratégias elegíveis de join. Estar habilitado não garante que toda consulta seja reescrita nem que todo skew seja resolvido.

Nesta aula há planos finais para alguns DataFrames e planos ainda não finais para outros; a investigação deve respeitar essa diferença. O broadcast automático foi desabilitado no experimento anterior, embora a habilitação geral do AQE continue verdadeira. Um diagnóstico completo requer plano final e métricas da action correspondente.


In [46]:
spark.conf.get("spark.sql.adaptive.enabled")

'true'

#### 12. Cache: materializar uma base para reutilização

`df_skew.cache()` registra a intenção de persistir o resultado para reutilização; não torna automaticamente todos os dados disponíveis em memória naquele instante. `show(1)` pede apenas uma amostra e **não garante materialização integral do cache**, mesmo que apareçam várias tasks no progresso.

O `count()` seguinte percorre o conjunto e retorna 2.000.000, sendo a action usada aqui para solicitar o processamento completo da base registrada em cache. Permanência e localização de cada bloco dependem dos recursos; os outputs não substituem uma inspeção do armazenamento.

`cache()` usa um nível padrão; `persist()` permite escolher explicitamente um `StorageLevel`. Não há chamada a `persist()` nesta aula. Cache pode evitar recomputar uma origem reutilizada, mas consome recursos e só compensa quando a reutilização justifica materialização e armazenamento.


In [47]:
df_cached = df_skew.cache()

In [48]:
df_cached.show(1)

+---+-----+
| id|grupo|
+---+-----+
|  0|    0|
+---+-----+
only showing top 1 row



In [49]:
df_cached.count()

2000000

##### 12.1. Identificar a reutilização no plano

A agregação `df_agg_cachead` usa a base cacheada e ordena as contagens por `grupo`. **Observado:** o plano otimizado contém `InMemoryRelation`, e o físico acessa `InMemoryTableScan [grupo]` com `StorageLevel(disk, memory, deserialized, 1 replicas)`.

Esse nível permite memória e disco; o nome `InMemory` não prova que todos os blocos estão em RAM. O `Range` desenhado sob a relação registra a origem do cache, sem significar necessariamente nova geração da base em cada acesso já materializado.

Cache não remove o trabalho posterior: permanecem `HashAggregate` parcial/final, `Exchange hashpartitioning` e `Exchange rangepartitioning`/`Sort`. Este bloco salva o plano, mas não a execução de `df_agg_cachead` nem uma comparação cronometrada com a consulta sem cache.


In [50]:
df_agg_cachead = (
    df_cached.groupBy("grupo")
    .agg(F.count("grupo").alias("contagem_grupo"))
    .orderBy(F.col("grupo"))
)

In [51]:
df_agg_cachead.explain(True)

== Parsed Logical Plan ==
'Sort ['grupo ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS contagem_grupo#311L]
   +- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
      +- Range (0, 2000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
grupo: bigint, contagem_grupo: bigint
Sort [grupo#74L ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS contagem_grupo#311L]
   +- Project [id#0L, CASE WHEN (id#0L < cast(1800000 as bigint)) THEN cast(0 as bigint) ELSE ((id#0L % cast(9 as bigint)) + cast(1 as bigint)) END AS grupo#74L]
      +- Range (0, 2000000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Sort [grupo#74L ASC NULLS FIRST], true
+- Aggregate [grupo#74L], [grupo#74L, count(grupo#74L) AS contagem_grupo#311L]
   +- Project [grupo#74L]
      +- InMemoryRelation [id#0L, grupo#74L], StorageLevel(disk, memory

##### 12.2. Liberar os blocos persistidos

`unpersist()` solicita a remoção dos blocos do cache; seu retorno continua sendo um DataFrame. Ele não apaga a definição lógica de `df_skew` nem impede uma nova recomputação. Liberar persistência após o uso evita reter recursos desnecessários; a chamada padrão não espera obrigatoriamente toda a remoção terminar antes de retornar.


In [52]:
df_cached.unpersist()

DataFrame[id: bigint, grupo: bigint]

#### 13. Spark UI: localizar a aplicação e investigar as actions

`spark.sparkContext.uiWebUrl` retorna o endereço da UI daquela aplicação: o hostname salvo pertence ao container da execução original e pode mudar. A URL localiza a interface, mas **não é uma captura de métricas**.

Para investigar os experimentos, a aba SQL permite relacionar uma action ao plano e aos jobs; Jobs/Stages ajuda a comparar tasks, volumes de shuffle, durações e spill; Storage permite examinar blocos persistidos enquanto existem. Tasks muito mais demoradas ou volumosas que as demais podem indicar skew, exigindo considerar também I/O e recursos.

Este notebook não conserva tabelas de tempos por task, bytes de shuffle ou spill. A consulta seguinte oferece uma action de agregação para investigação, sem autorizar inventar métricas da UI. Após `stop()`, a UI dessa aplicação pode deixar de estar disponível.

**Registro complementar do desafio 10:** o capítulo informa uma consulta de agregação da Parte B com SQL Query ID 23, duração registrada de 1 segundo e jobs 68, 69 e 70; a conclusão do estudante também relata spill zero em `HashAggregate`. Esses registros pertencem ao desafio, não à execução desta aula. Um tempo de consulta com agregação não isola o custo do join; sem métricas equivalentes do SortMergeJoin, eles não demonstram que broadcast foi mais rápido.


In [53]:
spark.sparkContext.uiWebUrl

'http://634a63abb3de:4041'

##### 13.1. Conferir novamente a distribuição por chave

A agregação final é feita sobre `df_skew` após `unpersist()`. **Observado:** o grupo 0 possui 1.800.000 linhas; os grupos 1 e 2 têm 22.223 cada e os demais têm 22.222. O total permanece em 2 milhões. Trata-se de contagem por valor da chave, não de distribuição por Spark Partition; não há ordenação solicitada neste `show()`.


In [54]:
df_skew.groupBy("grupo").count().show()

+-----+-------+
|grupo|  count|
+-----+-------+
|    0|1800000|
|    7|  22222|
|    6|  22222|
|    9|  22222|
|    5|  22222|
|    1|  22223|
|    3|  22222|
|    8|  22222|
|    2|  22223|
|    4|  22222|
+-----+-------+



#### 14. Encerrar a aplicação

`spark.stop()` encerra a sessão e o contexto. As estratégias estudadas precisam ser comparadas dentro de condições de execução conhecidas; encerrar a aplicação preserva no arquivo os outputs existentes, sem produzir um relatório de desempenho adicional.


In [55]:
spark.stop()

#### Resumo dos aprendizados

- Um filtro narrow reduziu o total de linhas e manteve oito partições; o agrupamento exigiu shuffle e agregação parcial/final.
- `repartition(4)` apresentou `Exchange RoundRobinPartitioning`; `coalesce(4)` apresentou `Coalesce`, ambos equilibrados nesta origem regular.
- Particionar por `grupo` concentrou 1.844.445 linhas numa partição. Round-robin distribuiu 250.000 por partição, preservando o skew da chave.
- Broadcast da dimensão de dez linhas evitou redistribuir a base por chave para o join; SortMergeJoin introduziu `Exchange` e `Sort` em ambos os lados no plano salvo.
- Cache é lazy, `show(1)` não comprova materialização total e persistência não elimina os shuffles seguintes. `InMemoryTableScan` evidencia o caminho de reutilização planejado.
- AQE habilitado e uma URL da Spark UI não comprovam otimização efetiva, duração menor ou ausência de spill. A aula documenta mecanismos e resultados de contagem, sem benchmark comparativo.

**Para aprofundar:** [capítulo 10 — Performance](../docs/10_performance.md), que também reúne as evidências parciais de Spark UI do desafio. **Próxima prática:** [desafio 10 — Performance](../desafios/desafio_10_performance.ipynb). Para revisar fundamentos relacionados, consulte [partições](../docs/05_particoes.md), [agregações](../docs/06_agregacoes.md) e [joins](../docs/07_joins.md).
